# C7-cnn-transfer — Practice p10 — Solution

The prefix filter freezes the stem's two parameterized children and all of `layer1` and `layer2`, while leaving `layer3`, `layer4`, and `fc` trainable.

In [ ]:
# Cache pin (course convention, plan 009): pretrained weights live in the repo's
# gitignored reference/cache/ -- resolve it from the repo root BEFORE importing torch.
import os, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "pyproject.toml").exists())
os.environ["TORCH_HOME"] = str(_root / "reference" / "cache" / "torch")

import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

# float32 register (course exception): pretrained resnet50 is a float32 artifact.
# No float64 default here; inputs are cast .to(torch.float32) at the model
# boundary; repeat float32 forwards are bit-identical.
SEED = 20260804

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval()
assert next(model.parameters()).dtype == torch.float32

frozen_prefixes = ("conv1", "bn1", "layer1", "layer2")
for name, p in model.named_parameters():
    if name.startswith(frozen_prefixes):
        p.requires_grad = False

n_frozen_tensors = sum(1 for p in model.parameters() if not p.requires_grad)
n_frozen_scalars = sum(p.numel() for p in model.parameters() if not p.requires_grad)
n_trainable_scalars = sum(p.numel() for p in model.parameters() if p.requires_grad)
model_total = sum(p.numel() for p in model.parameters())
split_ok = n_frozen_scalars + n_trainable_scalars == model_total
trainable_tops = sorted({name.split(".", 1)[0] for name, p in model.named_parameters() if p.requires_grad})


### Answer check

In [ ]:
assert n_frozen_tensors == 72
assert n_frozen_scalars == 1_444_928
assert n_trainable_scalars == 24_112_104
assert split_ok
assert trainable_tops == ["fc", "layer3", "layer4"]
